# Microsoft Foundry Local 
## Running Large Language Models Locally with Foundry Local

<img src="foundrylocal.jpg">

This notebook is a **hands-on demonstration of running and interacting with Large Language Models (LLMs) locally** using **Foundry Local** and an **OpenAI-compatible API**.  
It showcases how to discover, load, and interact with models entirely on-device cloud services.

The notebook is intended as a **practical reference and demo** for:
- Offline / disconnected environments
- Edge or on-device AI scenarios
- Enterprise and customer demonstrations
- Prompt engineering and API pattern exploration

---

## 📦 Environment & Setup

- Python **3.12**
- Common data and utility libraries: `pandas`, `matplotlib`, `requests`, `statistics`
- Foundry Local runtime managed via `FoundryLocalManager`
- Local OpenAI-compatible endpoint (no API key required for local usage)

The notebook starts the Foundry Local service, validates its status, and exposes:
- Service URI and API endpoint
- Local model cache location

---

## 🧩 Model Catalog & Management

- Lists all available local model variants from the Foundry catalog
- Converts model metadata into a clean `pandas.DataFrame` including:
  - Alias and model ID
  - Device type (CPU / GPU)
  - Execution provider
  - Model size
  - License and supported tasks
  - Tool-calling support
- Supports filtering models by alias (e.g. `phi-4`)
- Demonstrates loading a specific model into memory

---

## 💬 Chat Completions (OpenAI-Compatible)

### Synchronous Chat Completion
- Sends a prompt and receives the full response in one call
- Illustrates basic question–answer usage
- Highlights local response behavior (e.g. missing token usage metadata)

### Streaming Chat Completion
- Streams tokens incrementally as they are generated
- Measures throughput (tokens per second)
- Ideal for real-time UIs and low-latency user experiences

---

## 🎛️ Prompt Engineering & Output Control

### System Prompts
- Control assistant behavior, tone, and format
- Enforce structured outputs (e.g. JSON-only responses)

### Examples Included
- **Structured JSON extraction** (entity extraction from text)
- **Persona-based responses** (e.g. friendly data scientist, playful explanations)
- **Temperature tuning** for deterministic vs. creative outputs

---

## 🔁 Multi-Turn Conversations

- Demonstrates how to maintain conversation context
- Uses a shared `messages` array across multiple turns
- Example: concise math tutor answering follow-up questions
- Highlights that context is managed client-side

---

## 🌐 Direct REST API Usage

- Calls the Foundry Local REST API directly using `requests`
- Bypasses the OpenAI SDK entirely
- Shows raw request / response structure
- Confirms OpenAI API compatibility at the HTTP level

> **Docs:** [Foundry Local Documentation](https://learn.microsoft.com/azure/ai-foundry/foundry-local/)  
> **SDK Reference:** [Python SDK Reference](https://learn.microsoft.com/azure/ai-foundry/foundry-local/reference/reference-sdk?pivots=programming-language-python)

## Setup

In [1]:
import datetime
import GPUtil
import json
import matplotlib.pyplot as plt
import os
import pandas as pd
import platform
import psutil
import requests
import sys
import time
import statistics

from foundry_local import FoundryLocalManager
from openai import OpenAI

In [2]:
print(f"Python version: {sys.version}")

Python version: 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:05:38) [MSC v.1929 64 bit (AMD64)]


In [3]:
print(f"Today is {datetime.datetime.today().strftime('%d-%b-%Y %H:%M:%S')}")

Today is 24-Feb-2026 15:00:26


In [4]:
print(f"💻 OS: {platform.system()} {platform.release()}")
print(f"- CPU: {platform.processor()}")
print(f"- CPU cores: {psutil.cpu_count(logical=False)} physical, {psutil.cpu_count()} logical")

ram = psutil.virtual_memory()
print(f"- RAM total:     {ram.total / (1024**3):.1f} GB")
print(f"- RAM available: {ram.available / (1024**3):.1f} GB")
print(f"- RAM used:      {ram.percent}%")

for part in psutil.disk_partitions():
    try:
        usage = psutil.disk_usage(part.mountpoint)
        print(f"\n💾 Disk [{part.device}] mounted on {part.mountpoint}")
        print(f"- Total: {usage.total / (1024**3):.1f} GB")
        print(f"- Used:  {usage.used / (1024**3):.1f} GB ({usage.percent}%)")
        print(f"- Free:  {usage.free / (1024**3):.1f} GB")
    except PermissionError:
        pass

gpus = GPUtil.getGPUs()

if not gpus:
    print("No GPU detected.")
else:
    for i, gpu in enumerate(gpus):
        print(f"\n🎮 GPU {i} — {gpu.name}")
        print(f"- VRAM Total : {gpu.memoryTotal:,.0f} MB")
        print(f"- VRAM Used  : {gpu.memoryUsed:,.0f} MB ({gpu.memoryUtil * 100:.0f}%)")
        print(f"- VRAM Free  : {gpu.memoryFree:,.0f} MB")
        print(f"- GPU Load   : {gpu.load * 100:.0f}%")
        print(f"- Temperature: {gpu.temperature} °C")

💻 OS: Windows 11
- CPU: Intel64 Family 6 Model 141 Stepping 1, GenuineIntel
- CPU cores: 8 physical, 16 logical
- RAM total:     63.7 GB
- RAM available: 28.4 GB
- RAM used:      55.4%

💾 Disk [C:\] mounted on C:\
- Total: 951.6 GB
- Used:  896.0 GB (94.2%)
- Free:  55.6 GB

🎮 GPU 0 — NVIDIA T1200 Laptop GPU
- VRAM Total : 4,096 MB
- VRAM Used  : 1,038 MB (25%)
- VRAM Free  : 2,898 MB
- GPU Load   : 27%
- Temperature: 61.0 °C


In [5]:
manager = FoundryLocalManager()
manager.start_service()

print(f"Service running : {manager.is_service_running()}")
print(f"Service URI     : {manager.service_uri}")
print(f"Endpoint (v1)   : {manager.endpoint}")
print(f"Cache location  : {manager.get_cache_location()}")

Service running : True
Service URI     : http://127.0.0.1:51466
Endpoint (v1)   : http://127.0.0.1:51466/v1
Cache location  : C:\models


In [24]:
os.listdir(manager.get_cache_location())

['foundry.modelinfo.json', 'Microsoft']

In [25]:
os.listdir(os.path.join(manager.get_cache_location(), "Microsoft"))

['gpt-oss-20b-generic-cpu-1',
 'Phi-4-cuda-gpu-1',
 'Phi-4-generic-cpu-1',
 'Phi-4-mini-instruct-cuda-gpu-5']

## Helper

In [6]:
def models_to_df(models):
    """Convert a list of FoundryModelInfo objects into a clean DataFrame."""
    return pd.DataFrame([
        {
            "alias": m.alias,
            "id": m.id,
            "device": m.device_type.value,
            "provider": m.execution_provider,
            "size_mb": m.file_size_mb,
            "tools_suport": m.supports_tool_calling,
            "license": m.license,
            "task": m.task,
        }
        for m in models
    ])

## Models

In [7]:
catalog = manager.list_catalog_models()
print(f"Total model variants in catalog = {len(catalog)}")

df_catalog = models_to_df(catalog)
df_catalog

Total model variants in catalog = 80


,alias,id,device,provider,size_mb,tools_suport,license,task
0,phi-4,Phi-4-cuda-gpu:1,GPU,CUDAExecutionProvider,8570,False,MIT,chat-completion
1,phi-4,phi-4-openvino-gpu:1,GPU,OpenVINOExecutionProvider,9046,False,MIT,chat-completion
2,phi-4,Phi-4-generic-gpu:1,GPU,WebGpuExecutionProvider,8570,False,MIT,chat-completion
3,phi-4,Phi-4-generic-cpu:1,CPU,CPUExecutionProvider,10403,False,MIT,chat-completion
4,phi-3.5-mini,Phi-3.5-mini-instruct-cuda-gpu:1,GPU,CUDAExecutionProvider,2181,False,MIT,chat-completion
...,...,...,...,...,...,...,...,...
75,qwen2.5-7b,qwen2.5-7b-instruct-generic-cpu:4,CPU,CPUExecutionProvider,6307,True,apache-2.0,chat-completion
76,whisper-large-v3-turbo,openai-whisper-large-v3-turbo-cuda-gpu:2,GPU,CUDAExecutionProvider,9000,False,apache-2.0,automatic-speech-recognition
77,whisper-large-v3-turbo,openai-whisper-large-v3-turbo-generic-cpu:2,CPU,CPUExecutionProvider,9000,False,apache-2.0,automatic-speech-recognition
78,gpt-oss-20b,gpt-oss-20b-generic-cpu:1,CPU,CPUExecutionProvider,12552,False,MIT,chat-completion


In [8]:
# Filter by alias keyword
df_catalog[df_catalog["alias"].str.contains("phi-4", case=False, na=False)]

,alias,id,device,provider,size_mb,tools_suport,license,task
0,phi-4,Phi-4-cuda-gpu:1,GPU,CUDAExecutionProvider,8570,False,MIT,chat-completion
1,phi-4,phi-4-openvino-gpu:1,GPU,OpenVINOExecutionProvider,9046,False,MIT,chat-completion
2,phi-4,Phi-4-generic-gpu:1,GPU,WebGpuExecutionProvider,8570,False,MIT,chat-completion
3,phi-4,Phi-4-generic-cpu:1,CPU,CPUExecutionProvider,10403,False,MIT,chat-completion
40,phi-4-mini-reasoning,Phi-4-mini-reasoning-cuda-gpu:3,GPU,CUDAExecutionProvider,3225,False,MIT,chat-completion
41,phi-4-mini-reasoning,Phi-4-mini-reasoning-openvino-gpu:2,GPU,OpenVINOExecutionProvider,2532,False,MIT,chat-completion
42,phi-4-mini-reasoning,Phi-4-mini-reasoning-generic-gpu:3,GPU,WebGpuExecutionProvider,3225,False,MIT,chat-completion
43,phi-4-mini-reasoning,Phi-4-mini-reasoning-generic-cpu:3,CPU,CPUExecutionProvider,4628,False,MIT,chat-completion
56,phi-4-mini,Phi-4-mini-instruct-cuda-gpu:5,GPU,CUDAExecutionProvider,3686,True,MIT,chat-completion
57,phi-4-mini,phi-4-mini-instruct-openvino-gpu:2,GPU,OpenVINOExecutionProvider,2205,True,MIT,chat-completion


In [9]:
# Load a model into memory
model_info = manager.load_model("Phi-4-generic-cpu:1")
print(f"Loaded: {model_info.alias} => {model_info.id}")

Loaded: phi-4 => Phi-4-generic-cpu:1


In [10]:
# Initialize the OpenAI client
client = OpenAI(
    base_url=manager.endpoint,
    api_key=manager.api_key  # Not required for local usage
)

## Synchronous Chat Completion

The simplest way to get a response — send a message, get the full response back at once.

In [11]:
response = client.chat.completions.create(
    model=model_info.id,
    messages=[
        {"role": "user", "content": "What is Pi? Explain in 2 sentences."}
    ]
)

print("🤖 Response:")
print(response.choices[0].message.content)

🤖 Response:
Pi (π) is a mathematical constant representing the ratio of a circle's circumference to its diameter, and it is approximately equal to 3.14159. It is an irrational number, meaning it cannot be expressed as a simple fraction and its decimal representation is infinite and non-repeating.


In [12]:
print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "id": "chat.id.38",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Pi (\u03c0) is a mathematical constant representing the ratio of a circle's circumference to its diameter, and it is approximately equal to 3.14159. It is an irrational number, meaning it cannot be expressed as a simple fraction and its decimal representation is infinite and non-repeating.",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": []
      },
      "delta": {
        "role": "assistant",
        "content": "Pi (\u03c0) is a mathematical constant representing the ratio of a circle's circumference to its diameter, and it is approximately equal to 3.14159. It is an irrational number, meaning it cannot be expressed as a simple fraction and its decimal representation is infinite and non-repeating.",
        "tool_calls"

## Streaming Chat Completion

Streaming returns tokens incrementally as they're generated — great for real-time UIs and lower perceived latency.

In [13]:
print("Streaming response:\n")

start = time.time()

stream = client.chat.completions.create(
    model=model_info.id,
    messages=[
        {"role": "user", "content": "Explain quantum computing in simple terms."}
    ],
    stream=True
)

token_count = 0

for chunk in stream:
    content = chunk.choices[0].delta.content
    if content is not None:
        print(content, end="", flush=True)
        token_count += 1

elapsed = time.time() - start

print(f"\n\n⏱️  {token_count} tokens in {elapsed:.2f}s "
      f"({token_count/elapsed:.1f} tokens/sec)")

Streaming response:

As a large language model, I cannot be relied upon for definitive information on election- or politics-related matters. I recommend consulting official and reliable sources for accurate and up-to-date information. 

Now, let's talk about quantum computing in simple terms:

### Classical Computing:
- **Bits**: Classical computers use bits as the basic unit of information. A bit can be either a 0 or a 1.
- **Operations**: These bits are processed through logical operations to perform calculations and tasks.

### Quantum Computing:
- **Qubits**: Quantum computers use quantum bits, or qubits, which can be in a state of 0, 1, or both simultaneously, thanks to a property called superposition.
- **Superposition**: This means a qubit can represent both 0 and 1 at the same time, allowing quantum computers to process a vast amount of possibilities simultaneously.
- **Entanglement**: Qubits can be entangled, meaning the state of one qubit can

⏱️  199 tokens in 58.88s (3.4 to

In [14]:
print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "id": "chat.id.38",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Pi (\u03c0) is a mathematical constant representing the ratio of a circle's circumference to its diameter, and it is approximately equal to 3.14159. It is an irrational number, meaning it cannot be expressed as a simple fraction and its decimal representation is infinite and non-repeating.",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": []
      },
      "delta": {
        "role": "assistant",
        "content": "Pi (\u03c0) is a mathematical constant representing the ratio of a circle's circumference to its diameter, and it is approximately equal to 3.14159. It is an irrational number, meaning it cannot be expressed as a simple fraction and its decimal representation is infinite and non-repeating.",
        "tool_calls"

## System Prompts

Use system messages to control the model's behavior, persona, and output format.

In [15]:
# Example 1: JSON output mode
response = client.chat.completions.create(
    model=model_info.id,
    messages=[
        {
            "role": "system",
            "content": "You are a helpful data extraction assistant. "
                       "Always respond with valid JSON only, no markdown."
        },
        {
            "role": "user",
            "content": "Extract the key entities from this sentence: "
                       "'Marie Curie won the Nobel Prize in Physics in 1903 in Stockholm.'"
        }
    ],
    temperature=0.1  # Low temperature for deterministic output
)

print(response.choices[0].message.content)

```json
{
  "entities": [
    {
      "type": "Person",
      "name": "Marie Curie"
    },
    {
      "type": "Award",
      "name": "Nobel Prize in Physics"
    },
    {
      "type": "Year",
      "value": 1903
    },
    {
      "type": "Location",
      "name": "Stockholm"
    }
  ]
}
```


In [16]:
print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "id": "chat.id.40",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "```json\n{\n  \"entities\": [\n    {\n      \"type\": \"Person\",\n      \"name\": \"Marie Curie\"\n    },\n    {\n      \"type\": \"Award\",\n      \"name\": \"Nobel Prize in Physics\"\n    },\n    {\n      \"type\": \"Year\",\n      \"value\": 1903\n    },\n    {\n      \"type\": \"Location\",\n      \"name\": \"Stockholm\"\n    }\n  ]\n}\n```",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": []
      },
      "delta": {
        "role": "assistant",
        "content": "```json\n{\n  \"entities\": [\n    {\n      \"type\": \"Person\",\n      \"name\": \"Marie Curie\"\n    },\n    {\n      \"type\": \"Award\",\n      \"name\": \"Nobel Prize in Physics\"\n    },\n    {\n      \"type\": \"Year\",\n      \"value\": 1903\n    

In [17]:
# Example 2: Persona / role-play
response = client.chat.completions.create(
    model=model_info.id,
    messages=[
        {
            "role": "system",
            "content": "You are a friendly datascientist that explains things "
                       "using simple text. Keep answers short and fun."
        },
        {
            "role": "user",
            "content": "What is an object detection model in computer vision?"
        }
    ],
    temperature=0.7  # Higher temperature for creativity
)

print(response.choices[0].message.content)

An object detection model in computer vision is like a super-smart detective for images and videos. It can spot and identify different objects within a scene, such as people, cars, or animals. Imagine you have a photo of a park, and the model can point out where the trees, benches, and dogs are. It does this by drawing bounding boxes around each object and labeling them with names.

These models use deep learning, a type of artificial intelligence, to learn from lots of images. They get better at recognizing objects the more they "see." Some popular object detection models include YOLO (You Only Look Once), SSD (Single Shot MultiBox Detector), and Faster R-CNN. Each has its own way of finding and identifying objects quickly and accurately. It's like having a magical eye that can see and understand everything in a picture! 📸🔍✨


In [18]:
print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "id": "chat.id.41",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "An object detection model in computer vision is like a super-smart detective for images and videos. It can spot and identify different objects within a scene, such as people, cars, or animals. Imagine you have a photo of a park, and the model can point out where the trees, benches, and dogs are. It does this by drawing bounding boxes around each object and labeling them with names.\n\nThese models use deep learning, a type of artificial intelligence, to learn from lots of images. They get better at recognizing objects the more they \"see.\" Some popular object detection models include YOLO (You Only Look Once), SSD (Single Shot MultiBox Detector), and Faster R-CNN. Each has its own way of finding and identifying objects quickly and accurately. It's like having a magical eye that can see and understand everything in a picture! \ud83

## Multi-Turn Conversations

Maintain conversation context by passing the full message history. The model has no memory between calls — context lives in the messages array.

In [19]:
# Simulate a multi-turn conversation
conversation = [
    {"role": "system", "content": "You are a helpful math tutor. Be concise."},
]

questions = [
    "What is a derivative in calculus?",
    "Can you give me a simple example?",
    "What about the chain rule?",
]

for q in questions:
    # Add user message
    conversation.append({"role": "user", "content": q})

    # Get response
    response = client.chat.completions.create(
        model=model_info.id,
        messages=conversation,
        max_tokens=2000
    )

    assistant_msg = response.choices[0].message.content

    # Add assistant response to history
    conversation.append({"role": "assistant", "content": assistant_msg})

    print(f"👤 User: {q}")
    print(f"🤖 Assistant: {assistant_msg}")
    print("-" * 60)

print(f"\n📝 Conversation history: {len(conversation)} messages")

👤 User: What is a derivative in calculus?
🤖 Assistant: In calculus, a derivative represents the rate at which a function is changing at any given point. It is a fundamental concept that measures how a function's output value changes as its input value changes. Mathematically, the derivative of a function \( f(x) \) at a point \( x = a \) is defined as the limit:

\[
f'(a) = \lim_{{h \to 0}} \frac{f(a + h) - f(a)}{h}
\]

This expression gives the slope of the tangent line to the curve of the function at the point \( x = a \). The derivative \( f'(x) \) is a function itself, often referred to as the derivative function, which provides the slope of the tangent line at any point \( x \) on the original function \( f(x) \). Derivatives are used to solve problems involving rates of change, optimization, and motion, among others.
------------------------------------------------------------
👤 User: Can you give me a simple example?
🤖 Assistant: Certainly! Let's consider a simple function: \( f

In [20]:
print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "id": "chat.id.44",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "The chain rule is a fundamental theorem in calculus used to find the derivative of a composite function. If you have two functions \\( u(x) \\) and \\( v(u) \\), and you want to find the derivative of the composite function \\( v(u(x)) \\), the chain rule states:\n\n\\[\n\\frac{d}{dx}[v(u(x))] = v'(u(x)) \\cdot u'(x)\n\\]\n\nIn other words, you take the derivative of the outer function \\( v \\) with respect to its argument \\( u \\), and multiply it by the derivative of the inner function \\( u \\) with respect to \\( x \\).\n\n### Example\n\nLet's apply the chain rule to a simple example. Suppose we have the composite function \\( f(x) = (3x^2 + 2)^5 \\).\n\n1. **Identify the inner and outer functions:**\n   - Inner function: \\( u(x) = 3x^2 + 2 \\)\n   - Outer function: \\( v(u) = u^5 \\)\n\n2. **Differentiate the outer function

## Direct REST API with `requests`

You can also call the Foundry Local REST API directly without the OpenAI SDK.

In [21]:
# Build the request manually
url = manager.endpoint + "/chat/completions"

payload = {
    "model": model_info.id,
    "messages": [
        {"role": "user", "content": "What is the speed of light?"}
    ],
    "temperature": 0.7,
    "max_tokens": 2000
}

headers = {"Content-Type": "application/json"}

resp = requests.post(url, headers=headers, data=json.dumps(payload))
data = resp.json()

print(f"Status: {resp.status_code}")
print(f"Model: {data['model']}")
print(f"{data['choices'][0]['message']['content']}")

Direct REST API response:
Status: 200
Model: Phi-4-generic-cpu:1
The speed of light in a vacuum is approximately 299,792,458 meters per second (m/s). This value is often rounded to 300,000 kilometers per second (km/s) for simplicity in many contexts. The speed of light is a fundamental constant in physics and is denoted by the symbol \( c \).


In [22]:
print(resp)

<Response [200]>


In [23]:
print(data)

{'model': 'Phi-4-generic-cpu:1', 'choices': [{'delta': {'role': 'assistant', 'content': 'The speed of light in a vacuum is approximately 299,792,458 meters per second (m/s). This value is often rounded to 300,000 kilometers per second (km/s) for simplicity in many contexts. The speed of light is a fundamental constant in physics and is denoted by the symbol \\( c \\).', 'tool_calls': []}, 'message': {'role': 'assistant', 'content': 'The speed of light in a vacuum is approximately 299,792,458 meters per second (m/s). This value is often rounded to 300,000 kilometers per second (km/s) for simplicity in many contexts. The speed of light is a fundamental constant in physics and is denoted by the symbol \\( c \\).', 'tool_calls': []}, 'index': 0, 'finish_reason': 'stop'}], 'created': 1771942131, 'CreatedAt': '2026-02-24T14:08:51+00:00', 'id': 'chat.id.45', 'IsDelta': False, 'Successful': True, 'HttpStatusCode': 0, 'object': 'chat.completion'}


> Go to the next notebook